# Лабораторна 1 · Notebook 1 · Дослід до сервісу

**Питання частини:** чи допомагає LLM перетворити коротке повідомлення водія на корисний запис журналу несправностей, і які рішення все одно має приймати програма?

Сервісу ще немає. Цей notebook порівнює просте рішення на правилах і структуровану пропозицію LLM на тих самих зафіксованих повідомленнях, перевіряє пропозиції власним валідатором і формулює висновок для специфікації.

**Межа досліду:** тут немає HTTP-сервісу, сховища та залежності від нього. Код сервісу пізніше не імпортує цей notebook: до продукту переносяться лише перевірені правила й контракт.

## 1. Задача і формат запису

**Сервіс:** журнал несправностей авто. Механік бачить список записів, може створити запис вручну, а також надіслати коротке повідомлення водія, з якого AI-функція пропонує такий самий запис.

**Запис має 4 поля:**

| Поле | Тип | Правило | Чи визначається з повідомлення |
|---|---|---|---|
| `title` | рядок 3–80 | коротка назва несправності | так, з тексту повідомлення |
| `system` | одне з `battery`, `starter`, `fuel`, `ignition`, `immobilizer` | вузол, до якого належить несправність | так, якщо симптом описаний; інакше потрібне уточнення |
| `mileage_km` | ціле 0–2 000 000 або `null` | пробіг на момент звернення | лише якщо число назване в тексті; інакше `null` |
| `safety_stop` | `true` / `false` | ознаки небезпеки (дим, запах палива, іскри) | визначається детерміновано з тексту, а не моделлю |

**Що можна визначити з повідомлення:** `title` і `system` — коли водій описав симптом; `mileage_km` — тільки коли число є в тексті.
**Коли потрібне уточнення:** немає жодного симптому вузла («машина не їде»), або текст описує кілька різних причин.
**Що не вирішує модель:** `safety_stop` і сам факт збереження. Ознаки небезпеки перевіряє програма за фіксованим списком маркерів, і такий запис автоматично не зберігається.

In [1]:
import json
import os
import re

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

PROVIDER = "openai"
MODEL = "gpt-4o-mini"
TEMPERATURE = 0
SEED = 42

api_key = os.getenv("OPENAI_API_KEY", "")
client = OpenAI(api_key=api_key, timeout=30.0, max_retries=0) if api_key else None
print(f"provider={PROVIDER}; model={MODEL}; temperature={TEMPERATURE}; seed={SEED}; client_ready={client is not None}")

provider=openai; model=gpt-4o-mini; temperature=0; seed=42; client_ready=True


In [2]:
SYSTEMS = ["battery", "starter", "fuel", "ignition", "immobilizer"]
DANGER_MARKERS = ["дим", "пахне бензин", "запах бензин", "пахне палив", "іскр", "тече бензин"]
TITLE_MIN, TITLE_MAX = 3, 80
MILEAGE_MAX = 2_000_000

STATUSES = ["created", "needs_clarification", "safety_hold", "content_rejected", "invalid_proposal"]
print("Поля:", ["title", "system", "mileage_km", "safety_stop"])
print("Вузли:", SYSTEMS)
print("Можливі статуси рішення:", STATUSES)

Поля: ['title', 'system', 'mileage_km', 'safety_stop']
Вузли: ['battery', 'starter', 'fuel', 'ignition', 'immobilizer']
Можливі статуси рішення: ['created', 'needs_clarification', 'safety_hold', 'content_rejected', 'invalid_proposal']


## 2. Зафіксовані повідомлення та очікування

10 синтетичних повідомлень. **Очікування записані до того, як виконувався будь-який виклик моделі**: звичайні випадки, перефразування без ключових слів, неповний текст, ознака небезпеки і один випадок із заздалегідь хибною пропозицією моделі.

Для неоднозначного тексту очікування — `needs_clarification`, а не вигадане значення поля.

In [3]:
CASES = [
    {"id": "M01", "kind": "typical", "text": "Стартер лише клацає, світло на панелі яскраве, пробіг 120000 км",
     "expected": {"status": "created", "system": "starter", "mileage_km": 120000, "safety_stop": False}},
    {"id": "M02", "kind": "typical", "text": "Акумулятор сів, на панелі нічого не світиться",
     "expected": {"status": "created", "system": "battery", "mileage_km": None, "safety_stop": False}},
    {"id": "M03", "kind": "typical", "text": "Бак порожній, мотор крутить і не схоплює, 85 тис. км",
     "expected": {"status": "created", "system": "fuel", "mileage_km": 85000, "safety_stop": False}},
    {"id": "M04", "kind": "typical", "text": "Після мийки двигуна мокрі свічки, мотор крутить і не заводиться",
     "expected": {"status": "created", "system": "ignition", "mileage_km": None, "safety_stop": False}},
    {"id": "M05", "kind": "typical", "text": "Блимає значок імобілайзера, авто не реагує на ключ",
     "expected": {"status": "created", "system": "immobilizer", "mileage_km": None, "safety_stop": False}},
    {"id": "M06", "kind": "paraphrase", "text": "Після ночі на морозі ледь провертає, і все на панелі тьмяніє",
     "expected": {"status": "created", "system": "battery", "mileage_km": None, "safety_stop": False}},
    {"id": "M07", "kind": "paraphrase", "text": "Не чути гудіння насоса під заднім сидінням, мотор крутить і не хапає",
     "expected": {"status": "created", "system": "fuel", "mileage_km": None, "safety_stop": False}},
    {"id": "M08", "kind": "incomplete", "text": "Машина не їде, зробіть щось",
     "expected": {"status": "needs_clarification", "system": None, "mileage_km": None, "safety_stop": False}},
    {"id": "M09", "kind": "danger", "text": "Після спроби завести з-під капота йде дим і сильно пахне бензином",
     "expected": {"status": "safety_hold", "system": None, "mileage_km": None, "safety_stop": True}},
    # M10 перевіряє не модель, а чутливість валідатора: пропозиція структурно правильна,
    # але пробіг у ній не згаданий у тексті повідомлення.
    {"id": "M10", "kind": "wrong_proposal", "text": "Стартер клацає, світло на панелі яскраве",
     "fixed_proposal": {"title": "Стартер клацає", "system": "starter", "mileage_km": 150000,
                        "needs_clarification": False, "clarification_question": None},
     "expected": {"status": "content_rejected", "system": None, "mileage_km": None, "safety_stop": False}},
]

assert len(CASES) >= 8
assert {c["id"] for c in CASES}.__len__() == len(CASES)
[(c["id"], c["kind"], c["expected"]["status"]) for c in CASES]

[('M01', 'typical', 'created'),
 ('M02', 'typical', 'created'),
 ('M03', 'typical', 'created'),
 ('M04', 'typical', 'created'),
 ('M05', 'typical', 'created'),
 ('M06', 'paraphrase', 'created'),
 ('M07', 'paraphrase', 'created'),
 ('M08', 'incomplete', 'needs_clarification'),
 ('M09', 'danger', 'safety_hold'),
 ('M10', 'wrong_proposal', 'content_rejected')]

## 3. Просте рішення на правилах

Baseline шукає ключові слова вузлів і число пробігу регулярним виразом. Він навмисно простий і прозорий: там, де однозначного збігу немає, він повертає `needs_clarification`, а не вигадане значення.

In [4]:
SYSTEM_KEYWORDS = {
    "battery": ["акумулятор", "сів", "розряд"],
    "starter": ["стартер", "клацає"],
    "fuel": ["бак", "палив", "бензонасос"],
    "ignition": ["свічк", "запалюван", "котушк"],
    "immobilizer": ["імобілайзер", "чип", "ключ не"],
}


def detect_danger(text):
    """Детермінована ознака небезпеки. Модель на це рішення не впливає."""
    low = text.lower()
    return any(marker in low for marker in DANGER_MARKERS)


def extract_mileage(text):
    """Пробіг беремо лише з тексту: 120000, 120 000 км або 85 тис."""
    low = text.lower().replace(" ", " ")
    thousands = re.search(r"(\d{1,4})\s*тис", low)
    if thousands:
        return int(thousands.group(1)) * 1000
    plain = re.search(r"(\d[\d\s]{2,})\s*км", low)
    if plain:
        return int(re.sub(r"\s", "", plain.group(1)))
    return None


def propose_with_rules(text):
    low = text.lower()
    matched = [s for s, words in SYSTEM_KEYWORDS.items() if any(w in low for w in words)]
    system = matched[0] if len(matched) == 1 else None
    return {
        "title": text[:TITLE_MAX].strip(),
        "system": system,
        "mileage_km": extract_mileage(text),
        "needs_clarification": system is None,
        "clarification_question": None if system else "Який симптом спостерігається при спробі завести?",
    }


rules_proposals = {case["id"]: propose_with_rules(case["text"]) for case in CASES}
[(cid, p["system"], p["mileage_km"], p["needs_clarification"]) for cid, p in rules_proposals.items()]

[('M01', 'starter', 120000, False),
 ('M02', 'battery', None, False),
 ('M03', 'fuel', 85000, False),
 ('M04', 'ignition', None, False),
 ('M05', 'immobilizer', None, False),
 ('M06', None, None, True),
 ('M07', None, None, True),
 ('M08', None, None, True),
 ('M09', None, None, True),
 ('M10', 'starter', None, False)]

## 4. Валідатор пропозиції: контракт майбутнього сервісу

Валідатор не залежить від того, хто склав пропозицію — правила чи LLM. Порядок перевірок фіксований:

1. **Ознака небезпеки в тексті** → `safety_hold`, запис не створюється (рішення програми, не моделі).
2. **Схема:** типи, довжина `title`, `system` зі списку, `mileage_km` у межах → інакше `invalid_proposal`.
3. **Невизначеність:** `needs_clarification` або відсутні `title`/`system` → `needs_clarification`.
4. **Зміст:** `mileage_km` має бути згаданий у тексті повідомлення → інакше `content_rejected`.
5. Лише після цього — `created`.

Схема й зміст — різні перевірки: правильний тип поля ще не означає, що значення відповідає повідомленню.

In [5]:
def validate_proposal(text, proposal):
    """Повертає (status, reason, record|None). Той самий порядок перевірок перенесемо в сервіс."""
    if detect_danger(text):
        return "safety_hold", "danger_markers_in_text", None

    if not isinstance(proposal, dict):
        return "invalid_proposal", "proposal_is_not_object", None

    allowed_keys = {"title", "system", "mileage_km", "needs_clarification", "clarification_question"}
    if not set(proposal) <= allowed_keys:
        return "invalid_proposal", f"unexpected_keys:{sorted(set(proposal) - allowed_keys)}", None

    title = proposal.get("title")
    system = proposal.get("system")
    mileage = proposal.get("mileage_km")

    if title is not None and (not isinstance(title, str) or not TITLE_MIN <= len(title.strip()) <= TITLE_MAX):
        return "invalid_proposal", "title_length_or_type", None
    if system is not None and system not in SYSTEMS:
        return "invalid_proposal", f"system_not_allowed:{system}", None
    if mileage is not None and (not isinstance(mileage, int) or isinstance(mileage, bool) or not 0 <= mileage <= MILEAGE_MAX):
        return "invalid_proposal", "mileage_type_or_range", None

    if proposal.get("needs_clarification") or not title or not system:
        return "needs_clarification", "system_or_title_missing", None

    if mileage is not None and not mileage_supported_by_text(text, mileage):
        return "content_rejected", f"mileage_not_in_text:{mileage}", None

    record = {"title": title.strip(), "system": system, "mileage_km": mileage, "safety_stop": False}
    return "created", "ok", record


def mileage_supported_by_text(text, mileage):
    """Пробіг приймаємо лише тоді, коли те саме число справді є в повідомленні."""
    digits = re.sub(r"[^\d]", "", text)
    if str(mileage) in digits:
        return True
    if mileage % 1000 == 0 and str(mileage // 1000) in digits and "тис" in text.lower():
        return True
    return False


print(validate_proposal("Стартер клацає", {"title": "Стартер клацає", "system": "starter", "mileage_km": None,
                                           "needs_clarification": False, "clarification_question": None}))

('created', 'ok', {'title': 'Стартер клацає', 'system': 'starter', 'mileage_km': None, 'safety_stop': False})


## 5. Детерміновані тести валідатора

Ці перевірки не залежать від реальної моделі: пропозиції задані прямо в коді. Серед них — навмисно неправильні, щоб показати **чутливість перевірки**.

In [6]:
def _p(**kwargs):
    base = {"title": "Стартер клацає", "system": "starter", "mileage_km": None,
            "needs_clarification": False, "clarification_question": None}
    base.update(kwargs)
    return base


DETERMINISTIC_CHECKS = [
    ("правильна пропозиція зберігається",
     ("Стартер клацає, пробіг 120000 км", _p(mileage_km=120000)), "created"),
    ("пробіг, якого немає в тексті, відхиляється",
     ("Стартер клацає, світло яскраве", _p(mileage_km=150000)), "content_rejected"),
    ("вузол поза списком відхиляється",
     ("Стартер клацає", _p(system="engine_block")), "invalid_proposal"),
    ("порожній title не зберігається",
     ("Стартер клацає", _p(title=None)), "needs_clarification"),
    ("занадто довгий title відхиляється",
     ("Стартер клацає", _p(title="я" * 81)), "invalid_proposal"),
    ("рядок замість числа у mileage_km відхиляється",
     ("Стартер клацає, 120000 км", _p(mileage_km="120000")), "invalid_proposal"),
    ("невизначеність моделі не зберігається",
     ("Машина не їде", _p(system=None, needs_clarification=True)), "needs_clarification"),
    ("ознака небезпеки має пріоритет над пропозицією",
     ("Йде дим і пахне бензином", _p(system="fuel")), "safety_hold"),
    ("зайві поля в пропозиції відхиляються",
     ("Стартер клацає", {**_p(), "price": 100}), "invalid_proposal"),
]

failures = []
for name, (text, proposal), expected_status in DETERMINISTIC_CHECKS:
    status, reason, record = validate_proposal(text, proposal)
    if status != expected_status:
        failures.append((name, expected_status, status, reason))
    if expected_status != "created":
        assert record is None, f"{name}: відхилена пропозиція не повинна давати запис"

print(f"deterministic checks: {len(DETERMINISTIC_CHECKS) - len(failures)} passed, {len(failures)} failed")
assert not failures, failures

deterministic checks: 9 passed, 0 failed


## 6. Пропозиція LLM на тих самих повідомленнях

Модель отримує повідомлення й схему пропозиції. Вона **не вирішує**, чи зберігати запис: її пропозицію перевіряє той самий валідатор.

`PROMPT_A` — базова інструкція. Якщо ключ не налаштовано, жива частина позначається `not_run`, а детерміновані перевірки вище лишаються виконаними.

In [7]:
PROMPT_A = (
    "You turn a short Ukrainian driver message into a structured car fault record.\n"
    f"Allowed system values: {SYSTEMS}.\n"
    "Return one JSON object with keys: title (3-80 chars), system, mileage_km (integer or null), "
    "needs_clarification (boolean), clarification_question (string or null).\n"
    "Set system to null and needs_clarification to true when the message names no symptom of a specific system."
)

# Єдина відмінність PROMPT_B: одне додаткове речення про пробіг.
PROMPT_B = PROMPT_A + "\nSet mileage_km to null unless the message explicitly states the mileage number."

LIVE_CASES = [case for case in CASES if case["kind"] != "wrong_proposal"]


def ask_model(prompt, text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": prompt}, {"role": "user", "content": text}],
        response_format={"type": "json_object"},
        temperature=TEMPERATURE,
        seed=SEED,
    )
    return json.loads(response.choices[0].message.content or "{}")


def run_variant(prompt):
    """Повертає {case_id: {"proposal": ..., "status": ..., "reason": ..., "record": ...}}"""
    out = {}
    for case in LIVE_CASES:
        proposal = ask_model(prompt, case["text"])
        status, reason, record = validate_proposal(case["text"], proposal)
        out[case["id"]] = {"proposal": proposal, "status": status, "reason": reason, "record": record}
    return out


llm_a = {}
live_status = "not_run"
if client is None:
    print("Жива частина: not_run (OPENAI_API_KEY не налаштовано). Детерміновані перевірки вище виконано.")
else:
    try:
        llm_a = run_variant(PROMPT_A)
        live_status = "executed"
        print(f"Жива частина виконана: {PROVIDER}/{MODEL}, {len(llm_a)} випадків")
    except Exception as error:
        print("Live LLM failed:", type(error).__name__, getattr(error, "status_code", None))

{cid: (data["status"], data["proposal"].get("system"), data["proposal"].get("mileage_km")) for cid, data in llm_a.items()}

Жива частина виконана: openai/gpt-4o-mini, 9 випадків


{'M01': ('created', 'starter', 120000),
 'M02': ('created', 'battery', None),
 'M03': ('created', 'fuel', 85000),
 'M04': ('created', 'ignition', None),
 'M05': ('created', 'immobilizer', None),
 'M06': ('needs_clarification', None, None),
 'M07': ('created', 'fuel', None),
 'M08': ('needs_clarification', None, None),
 'M09': ('safety_hold', 'fuel', None)}

## 7. Таблиця результатів по випадках

Для правил і для LLM застосовано **той самий валідатор і ті самі очікування**. Випадок `M10` перевіряє заздалегідь хибну пропозицію, тому для живої моделі він не застосовується.

In [8]:
from IPython.display import Markdown, display


def decide(case, proposals_by_id):
    if case["kind"] == "wrong_proposal":
        proposal = case["fixed_proposal"]
    else:
        entry = proposals_by_id.get(case["id"])
        if entry is None:
            return None
        proposal = entry["proposal"] if isinstance(entry, dict) and "proposal" in entry else entry
    status, reason, record = validate_proposal(case["text"], proposal)
    return {"status": status, "reason": reason, "record": record}


def matches(case, outcome):
    if outcome is None:
        return None
    expected = case["expected"]
    if outcome["status"] != expected["status"]:
        return False
    if outcome["status"] == "created":
        record = outcome["record"]
        return record["system"] == expected["system"] and record["mileage_km"] == expected["mileage_km"]
    return True


rows = []
for case in CASES:
    rules_outcome = decide(case, rules_proposals)
    llm_outcome = decide(case, llm_a) if (llm_a or case["kind"] == "wrong_proposal") else None
    rows.append({
        "id": case["id"], "kind": case["kind"], "expected": case["expected"]["status"],
        "rules": rules_outcome["status"], "rules_ok": matches(case, rules_outcome),
        "llm": llm_outcome["status"] if llm_outcome else "not_run",
        "llm_ok": matches(case, llm_outcome),
        "reason": (llm_outcome or rules_outcome)["reason"],
    })

table = ["| id | тип | очікування | правила | ✓ | LLM | ✓ | причина (LLM) |", "|---|---|---|---|:-:|---|:-:|---|"]
for row in rows:
    mark = lambda v: "—" if v is None else ("✅" if v else "❌")
    table.append(
        f"| {row['id']} | {row['kind']} | {row['expected']} | {row['rules']} | {mark(row['rules_ok'])} "
        f"| {row['llm']} | {mark(row['llm_ok'])} | {row['reason']} |"
    )
display(Markdown("\n".join(table)))

summary = {
    "live_llm": live_status,
    "cases": len(CASES),
    "rules_ok": sum(1 for r in rows if r["rules_ok"]),
    "llm_ok": sum(1 for r in rows if r["llm_ok"]) if llm_a else None,
    "deterministic_checks_passed": len(DETERMINISTIC_CHECKS),
}
summary

| id | тип | очікування | правила | ✓ | LLM | ✓ | причина (LLM) |
|---|---|---|---|:-:|---|:-:|---|
| M01 | typical | created | created | ✅ | created | ✅ | ok |
| M02 | typical | created | created | ✅ | created | ✅ | ok |
| M03 | typical | created | created | ✅ | created | ✅ | ok |
| M04 | typical | created | created | ✅ | created | ✅ | ok |
| M05 | typical | created | created | ✅ | created | ✅ | ok |
| M06 | paraphrase | created | needs_clarification | ❌ | needs_clarification | ❌ | system_or_title_missing |
| M07 | paraphrase | created | needs_clarification | ❌ | created | ✅ | ok |
| M08 | incomplete | needs_clarification | needs_clarification | ✅ | needs_clarification | ✅ | system_or_title_missing |
| M09 | danger | safety_hold | safety_hold | ✅ | safety_hold | ✅ | danger_markers_in_text |
| M10 | wrong_proposal | content_rejected | content_rejected | ✅ | content_rejected | ✅ | mileage_not_in_text:150000 |

{'live_llm': 'executed',
 'cases': 10,
 'rules_ok': 8,
 'llm_ok': 9,
 'deterministic_checks_passed': 9}

## 8. Контрольований експеримент: дві інструкції

Набір випадків і спосіб оцінювання лишаються тими самими. **Єдина відмінність** `PROMPT_B` — одне додаткове речення: `mileage_km` має бути `null`, якщо числа пробігу в повідомленні немає.

Показник, на який впливає ця зміна: кількість випадків, де модель вказала пробіг, якого немає в тексті (`content_rejected` із причиною `mileage_not_in_text`).

In [9]:
experiment = {"variant_a": None, "variant_b": None}

if not llm_a:
    print("Експеримент: not_run (немає живих відповідей для варіанта A).")
else:
    llm_b = run_variant(PROMPT_B)

    def variant_stats(results):
        invented = [cid for cid, data in results.items() if data["reason"].startswith("mileage_not_in_text")]
        ok = sum(1 for case in LIVE_CASES if matches(case, decide(case, results)))
        return {"cases": len(results), "expected_status_match": ok,
                "invented_mileage": len(invented), "invented_cases": invented}

    experiment = {"variant_a": variant_stats(llm_a), "variant_b": variant_stats(llm_b)}
    display(Markdown(
        "| варіант | відмінність | збіг зі очікуванням | вигаданий пробіг |\n|---|---|:-:|:-:|\n"
        f"| A | базова інструкція | {experiment['variant_a']['expected_status_match']}/{len(LIVE_CASES)} "
        f"| {experiment['variant_a']['invented_mileage']} |\n"
        f"| B | + одне речення про null | {experiment['variant_b']['expected_status_match']}/{len(LIVE_CASES)} "
        f"| {experiment['variant_b']['invented_mileage']} |"
    ))

experiment

| варіант | відмінність | збіг зі очікуванням | вигаданий пробіг |
|---|---|:-:|:-:|
| A | базова інструкція | 8/9 | 0 |
| B | + одне речення про null | 8/9 | 0 |

{'variant_a': {'cases': 9,
  'expected_status_match': 8,
  'invented_mileage': 0,
  'invented_cases': []},
 'variant_b': {'cases': 9,
  'expected_status_match': 8,
  'invented_mileage': 0,
  'invented_cases': []}}

## 9. Висновок для специфікації

**Фактичні результати цього прогону:** детерміновані перевірки валідатора — 9/9 passed; правила — 8/10 випадків; жива LLM (`openai/gpt-4o-mini`, `temperature=0`, `seed=42`) — 9/10.

**Де правил достатньо.** На всіх п'яти типових повідомленнях із ключовими словами (`M01`–`M05`) правила дали той самий результат, що й модель, включно з пробігом `120000` і `85 тис.`. Ознаку небезпеки (`M09`) і хибну пропозицію з вигаданим пробігом (`M10`) теж опрацював детермінований код, а не модель.

**Де LLM допомогла.** `M07` («не чути гудіння насоса під заднім сидінням») — правила не знайшли ключового слова й повернули `needs_clarification`, а модель правильно визначила `fuel`. Це єдиний випадок, де LLM дала перевагу на цьому наборі.

**Де LLM помилилася.** `M06` («після ночі на морозі ледь провертає, все тьмяніє») — очікувався `battery`, але модель повернула `needs_clarification`. Очікування не змінюю: це помилка моделі, а не помилка набору випадків. Помилка безпечна — вона веде до уточнення, а не до неправильного запису.

**Результат контрольованого експерименту.** Варіанти інструкції A і B відрізнялися одним реченням («`mileage_km` має бути `null`, якщо числа немає в тексті»). На тих самих 9 випадках і з тим самим способом оцінювання результат ідентичний: 8/9 збігів і **0 вигаданих пробігів в обох варіантах**. Тобто на цьому наборі додаткове речення не дало вимірюваного ефекту: модель і без нього не вигадувала пробіг. Це не доводить, що воно зайве на інших входах — доказу просто немає.

**Що переноситься в специфікацію сервісу.**

1. **Порядок перевірок** фіксований і не залежить від того, хто склав пропозицію: ознака небезпеки в тексті → схема → невизначеність → зміст → збереження.
2. **Ознаку небезпеки визначає програма** за фіксованим списком маркерів. Такий запис не зберігається автоматично навіть тоді, коли пропозиція моделі структурно правильна (`M09`: модель повернула `system: fuel`, рішення — `safety_hold`).
3. **`system` лише зі списку п'яти значень**; будь-яке інше значення — `invalid_proposal`.
4. **`mileage_km` приймається лише тоді, коли те саме число є в повідомленні.** Схема тут не допомагає: у `M10` тип поля правильний, а значення суперечить тексту.
5. **Невизначеність не зберігається:** `needs_clarification` від моделі або відсутні `title`/`system` дають статус із питанням, а не запис.
6. **Модель не має прямого доступу до сховища:** сервіс приймає або відхиляє її пропозицію, а ручний шлях працює без моделі.
7. **Статуси рішення** для HTTP-контракту: `created`, `needs_clarification`, `safety_hold`, `content_rejected`, `invalid_proposal`, плюс окремий результат недоступності моделі.

**Чого цей дослід не доводить.** Він не перевіряє HTTP-сервіс, сховище та інтеграцію провайдера з готовим продуктом: 10 синтетичних випадків, один прогін, один набір формулювань. Жива перевірка інтеграції через API — завдання другого notebook.